# Milestone 1, Task 2: Cleaned Dataset Artifact
**Team:** Pharma 1A (Break Through Tech AI Studio, Fall 2026) | **Host company:** WebMD | **Challenge Advisor:** Kimia Naeiji | **AI Studio Coach:** Anushka Naik

**Target variables (from the Challenge Project Overview):**
- **Primary: `total_spend`**, evaluated with MAPE (goal under 30%) and R² (goal above 0.55) on the held-out H2 set. We model `log_total_spend` and convert back.
- **Secondary: `cpe`**, the closest column to the `meta_mapped_cpe` named in the overview (to be confirmed with our Challenge Advisor).

This notebook applies the fixes proposed in `01_data_quality_report.ipynb`, defines the feature set for each target, and produces three files:

| File | What it is | Use it for |
|---|---|---|
| `data/processed/cleaned_df.csv` | Cleaned, human-readable data. Text categories kept, flags added, no nulls. | EDA, baseline statistics, the Task 4 write-up |
| `data/processed/cleaned_df_encoded.csv` | Model-ready version with one-hot encoded categories and raw text removed | All modeling |
| `reports/feature_sets.json` | The exact feature columns, row filters and excluded columns for each target | Loading the right columns in every modeling notebook |

Every step is labeled with the issue ID it resolves (DQ-xx) so instructors can audit each decision. The raw file is never modified.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW_PATH = "../data/BreakThroughTech_training_data_1.xlsx"
OUT_CLEAN = "../data/processed/cleaned_df.csv"
OUT_ENCODED = "../data/processed/cleaned_df_encoded.csv"
OUT_FEATURES = "../reports/feature_sets.json"

raw = pd.read_excel(RAW_PATH)
df = raw.copy()
print("Raw shape:", raw.shape)

Raw shape: (417, 28)


## Step 1. Brand standardization (DQ-01)
Each raw `brand` value is mapped to three clean columns:
- **company:** one standard name per advertiser (e.g. `BMS` and `BRISTOL-MYERS SQUIBB` both become `BRISTOL MYERS SQUIBB`)
- **product:** the drug name when the raw value contains one, otherwise `unspecified`
- **channel_tag:** trailing vendor or channel codes (`MEDSCAPE`, `MDALERT`, `ZS`, `BA`, `EA`) or indication tags (`PSA`, `PSO`), otherwise `none`

Decisions worth noting:
- Janssen sub-units (`JANSSEN BIOTECH`, `JANSSEN CNS`, `JANSSEN PHARMACEUTICALS`) and `J&J` all map to `JANSSEN`, since Janssen is J&J's pharmaceutical company.
- Acquired companies stay under the name used in the data (`SEAGEN`, `HORIZON`) because the campaigns were run under those names. The one exception is `AMGEN_HORIZON_KRYSTEXXA`, which already names Amgen as the advertiser.
- The mapping is written out explicitly (not with a regex) so every choice is visible, and the code stops with an error if a new unmapped brand ever appears.

In [2]:
BRAND_MAP = {
    # raw value: (company, product, channel_tag)
    "ABBVIE": ("ABBVIE", None, None),
    "ADC THERAPEUTICS": ("ADC THERAPEUTICS", None, None),
    "ADC_THERAPEUTICS_ZYNLONTA": ("ADC THERAPEUTICS", "ZYNLONTA", None),
    "AMGEN": ("AMGEN", None, None),
    "AMGEN_HORIZON_KRYSTEXXA": ("AMGEN", "KRYSTEXXA", None),
    "ASTELLAS": ("ASTELLAS", None, None),
    "ASTELLAS_XTANDI": ("ASTELLAS", "XTANDI", None),
    "ASTRAZENECA_ENHERTU": ("ASTRAZENECA", "ENHERTU", None),
    "BMS": ("BRISTOL MYERS SQUIBB", None, None),
    "BRISTOL-MYERS SQUIBB": ("BRISTOL MYERS SQUIBB", None, None),
    "BOEHRINGER INGELHEIM": ("BOEHRINGER INGELHEIM", None, None),
    "BOEHRINGER-INGELHEIM": ("BOEHRINGER INGELHEIM", None, None),
    "CALLIDITAS": ("CALLIDITAS", None, None),
    "DAY ONE BIOPHARMACEUTICALS": ("DAY ONE BIOPHARMACEUTICALS", None, None),
    "DAY_ONE_BIO": ("DAY ONE BIOPHARMACEUTICALS", None, None),
    "EISAI": ("EISAI", None, None),
    "EMD SERONO": ("EMD SERONO", None, None),
    "SERONO": ("EMD SERONO", None, None),
    "SERONO_TEPMETKO": ("EMD SERONO", "TEPMETKO", None),
    "FERRING": ("FERRING", None, None),
    "GILEAD": ("GILEAD", None, None),
    "GSK": ("GSK", None, None),
    "HORIZON": ("HORIZON", None, None),
    "HORIZON THERAPEUTICS": ("HORIZON", None, None),
    "INTRA-CELLULAR THERAPIES": ("INTRA-CELLULAR THERAPIES", None, None),
    "J&J": ("JANSSEN", None, None),
    "JANSSEN": ("JANSSEN", None, None),
    "JANSSEN BIOTECH": ("JANSSEN", None, None),
    "JANSSEN CNS": ("JANSSEN", None, None),
    "JANSSEN PHARMACEUTICALS": ("JANSSEN", None, None),
    "JANSSEN_DARZALEX": ("JANSSEN", "DARZALEX", None),
    "JANSSEN_DARZALEX_MDALERT": ("JANSSEN", "DARZALEX", "MDALERT"),
    "JANSSEN_ERLEADA": ("JANSSEN", "ERLEADA", None),
    "JANSSEN_INVEGA": ("JANSSEN", "INVEGA", None),
    "JANSSEN_RYBREVANT": ("JANSSEN", "RYBREVANT", None),
    "JANSSEN_SPRAVATO": ("JANSSEN", "SPRAVATO", None),
    "JANSSEN_TALVEY": ("JANSSEN", "TALVEY", None),
    "JANSSEN_TECVAYLI": ("JANSSEN", "TECVAYLI", None),
    "JANSSEN_TREMFYAPSA": ("JANSSEN", "TREMFYA", "PSA"),
    "JANSSEN_TREMFYAPSO": ("JANSSEN", "TREMFYA", "PSO"),
    "JAZZ PHARMACEUTICALS": ("JAZZ PHARMACEUTICALS", None, None),
    "JAZZ_ZEPZELCA_BA": ("JAZZ PHARMACEUTICALS", "ZEPZELCA", "BA"),
    "JAZZ_ZEPZELCA_EA": ("JAZZ PHARMACEUTICALS", "ZEPZELCA", "EA"),
    "LILLY": ("LILLY", None, None),
    "LILLY_MOUNJARO_MDALERT": ("LILLY", "MOUNJARO", "MDALERT"),
    "LILLY_MOUNJARO_MEDSCAPE": ("LILLY", "MOUNJARO", "MEDSCAPE"),
    "MERCK": ("MERCK", None, None),
    "NEUROCRINE": ("NEUROCRINE", None, None),
    "NOVARTIS": ("NOVARTIS", None, None),
    "NOVO NORDISK": ("NOVO NORDISK", None, None),
    "NOVONORDISK_OZEMPIC": ("NOVO NORDISK", "OZEMPIC", None),
    "NOVONORDISK_WEGOVY": ("NOVO NORDISK", "WEGOVY", None),
    "OTSUKA PHARMACEUTICAL": ("OTSUKA", None, None),
    "PFIZER": ("PFIZER", None, None),
    "PFIZER_ELREXFIO": ("PFIZER", "ELREXFIO", None),
    "PFIZER_PAXLOVID": ("PFIZER", "PAXLOVID", None),
    "PROACT": ("PROACT", None, None),
    "RIGEL": ("RIGEL", None, None),
    "RIGEL PHARMACEUTICALS": ("RIGEL", None, None),
    "SEAGEN": ("SEAGEN", None, None),
    "SEAGEN_ADCETRIS_MDALERT": ("SEAGEN", "ADCETRIS", "MDALERT"),
    "SEAGEN_ADCETRIS_MEDSCAPE": ("SEAGEN", "ADCETRIS", "MEDSCAPE"),
    "SEAGEN_TIVDAK": ("SEAGEN", "TIVDAK", None),
    "SEAGEN_TIVDAK_MDALERT": ("SEAGEN", "TIVDAK", "MDALERT"),
    "SERVIER": ("SERVIER", None, None),
    "STEMLINE": ("STEMLINE", None, None),
    "SUMITOMO PHARMA": ("SUMITOMO PHARMA", None, None),
    "SUMITOMO_GEMTESA": ("SUMITOMO PHARMA", "GEMTESA", None),
    "TAKEDA": ("TAKEDA", None, None),
    "TAKEDA_ZS": ("TAKEDA", None, "ZS"),
    "TEVA": ("TEVA", None, None),
    "VIIV": ("VIIV HEALTHCARE", None, None),
    "VIIV HEALTHCARE": ("VIIV HEALTHCARE", None, None),
}

unmapped = set(df["brand"].unique()) - set(BRAND_MAP)
assert not unmapped, f"Unmapped brand values: {unmapped}"

df = df.rename(columns={"brand": "brand_raw"})
df["company"] = df["brand_raw"].map(lambda b: BRAND_MAP[b][0])
df["product"] = df["brand_raw"].map(lambda b: BRAND_MAP[b][1] or "unspecified")
df["channel_tag"] = df["brand_raw"].map(lambda b: BRAND_MAP[b][2] or "none")

print(f"Raw brand values: {raw['brand'].nunique()}  ->  companies: {df['company'].nunique()}")
df["company"].value_counts().to_frame("rows")

Raw brand values: 73  ->  companies: 34


,rows
company,
MERCK,107
LILLY,42
BRISTOL MYERS SQUIBB,35
SEAGEN,26
BOEHRINGER INGELHEIM,22
JANSSEN,22
TAKEDA,17
GILEAD,14
NOVARTIS,11


## Step 2. Country, pillar and media type (DQ-03, DQ-04, DQ-06, DQ-07, DQ-08)

In [3]:
# Flag the structural block BEFORE filling anything, so the pattern is preserved
df["targeting_info_missing"] = raw[["country", "pillar", "media_type", "profession", "specialty"]].isna().all(axis=1).astype(int)

COUNTRY_NAMES = {"USA": "United States", "ZAF": "South Africa", "VGB": "British Virgin Islands",
                 "VIR": "U.S. Virgin Islands", "ZWE": "Zimbabwe", "ZMB": "Zambia",
                 "VNM": "Vietnam", "IND": "India", "UNKNOWN": "Unknown"}
df["country"] = df["country"].str.upper().fillna("UNKNOWN")
df["country_name"] = df["country"].map(COUNTRY_NAMES)
assert df["country_name"].notna().all()

df["pillar"] = df["pillar"].fillna("unknown")
df["media_type"] = df["media_type"].str.lower().fillna("unknown")

for c in ["country", "pillar", "media_type"]:
    print(df[c].value_counts().to_dict())

{'USA': 201, 'ZAF': 95, 'UNKNOWN': 56, 'VGB': 32, 'VIR': 18, 'ZWE': 5, 'ZMB': 4, 'VNM': 3, 'IND': 3}
{'search': 235, 'reference and tools': 97, 'unknown': 56, 'news and perspectives': 27, 'other': 2}
{'editorial_videos': 243, 'sponsored_videos': 94, 'unknown': 80}


## Step 3. Profession and specialty (DQ-09 to DQ-13)
Numeric codes are converted to labels like `code_8`. They are nominal IDs with no natural ordering (code 99 is not "more" than code 8), so they must be treated as categories. `undefined` in specialty is a hidden null and becomes `unknown`.

In [4]:
def to_label(v):
    if pd.isna(v):
        return "unknown"
    if isinstance(v, (int, float, np.integer, np.floating)):
        return f"code_{int(v)}"
    v = str(v).strip().lower()
    if v.isdigit():
        return f"code_{v}"
    return "unknown" if v == "undefined" else v

df["profession"] = raw["profession"].map(to_label)
df["specialty"] = raw["specialty"].map(to_label)
df["has_specialty"] = (df["specialty"] != "unknown").astype(int)

print(df["profession"].value_counts().to_dict())
print(df["specialty"].value_counts().to_dict())

{'code_8': 162, 'code_99': 107, 'unknown': 56, 'psychologist': 45, 'physician assistant': 16, 'code_19': 6, 'code_20': 5, 'code_11': 4, 'physician': 4, 'code_15': 3, 'code_10': 3, 'code_12': 3, 'code_23': 2, 'code_13': 1}
{'unknown': 352, 'urology': 46, 'vascular medicine': 8, 'code_77': 3, 'code_35': 2, 'code_45': 2, 'code_59': 2, 'code_70': 1, 'code_32': 1}


## Step 4. Campaign dates (DQ-14, DQ-15)
- `campaign_start_date` is parsed from text to a real date. The 3 missing values are filled with the median start date for the same company, and flagged.
- `campaign_end_date` is **dropped**: 414 rows hold the 2099-12-31 placeholder and 3 are null, so it carries no information.
- Time features are built from the start date. `campaign_age_days` is measured from the latest start date in the data (2026-04-17) so the value is reproducible and does not change depending on when the notebook is run.

In [5]:
start = pd.to_datetime(raw["campaign_start_date"].str.replace(" UTC", "", regex=False), errors="coerce")
df["start_date_imputed"] = start.isna().astype(int)
company_median = start.groupby(df["company"]).transform("median")
start = start.fillna(company_median).fillna(start.median())
df["campaign_start_date"] = start.dt.normalize()

REFERENCE_DATE = start.max().normalize()
df["start_year"] = start.dt.year
df["start_quarter"] = start.dt.quarter
df["start_month"] = start.dt.month
df["campaign_age_days"] = (REFERENCE_DATE - df["campaign_start_date"]).dt.days

df = df.drop(columns=["campaign_end_date"])
print("Reference date:", REFERENCE_DATE.date())
print(df.loc[df.start_date_imputed == 1, ["company", "sub_product_name", "campaign_start_date"]])
print(df["start_year"].value_counts().sort_index().to_dict())

Reference date: 2026-04-17
    company   sub_product_name campaign_start_date
170   LILLY          BRANDSPOT          2023-11-16
183   LILLY  TARGET LIST MEDIA          2023-11-16
282   MERCK           INFOSITE          2024-10-22
{2020: 1, 2021: 6, 2022: 31, 2023: 120, 2024: 142, 2025: 108, 2026: 9}


## Step 5. Cost columns and the targets (DQ-18, DQ-30, DQ-32, DQ-33)
- **cpe:** filled from `total_budget_spent / total_engagements`, which is exactly how cpe relates to budget in every row where both exist.
- **total_spend:** the 3 blanks are filled from `actual_spend_per_engagement x total_engagements` so the file has no gaps. That rebuild equals the column median, meaning the source had already median-filled these rows, so their true spend is unknown. `total_spend_imputed` marks them and they are **excluded from training and evaluation** for the spend target.
- **spend_near_zero:** marks the 8 campaigns with total_spend under 1,000 (2 are exactly 0). They are **excluded when calculating MAPE**, since MAPE is undefined at 0 and explodes near it.
- **budget_anomaly:** flags the rows where budget is $0 or far below cpe x engagements (7 rows), plus 1 row where the imputed cpe comes out at $0.02, far below any product rate.

In [6]:
df["cpe_imputed"] = df["cpe"].isna().astype(int)
df.loc[df["cpe"].isna(), "cpe"] = (df["total_budget_spent"] / df["total_engagements"]).round(2)

df["total_spend_imputed"] = df["total_spend"].isna().astype(int)
df.loc[df["total_spend"].isna(), "total_spend"] = (df["actual_spend_per_engagement"] * df["total_engagements"]).round(2)
df["spend_near_zero"] = (df["total_spend"] < 1000).astype(int)

cpe_orig = raw["cpe"]
mismatch = (raw["total_engagements"] * cpe_orig - raw["total_budget_spent"]).abs() > 1
implied_cpe_tiny = df["cpe_imputed"].eq(1) & (df["cpe"] < 1)  # imputed cpe far below every product rate
df["budget_anomaly"] = ((raw["total_budget_spent"] == 0) | mismatch | implied_cpe_tiny).astype(int)

print("cpe imputed:", df.cpe_imputed.sum(), "| values used:", df.loc[df.cpe_imputed == 1, "cpe"].value_counts().to_dict())
print("total_spend imputed (excluded from spend training and evaluation):", df.total_spend_imputed.sum())
print("spend_near_zero (excluded from MAPE):", df.spend_near_zero.sum())
print("budget_anomaly rows:", df.budget_anomaly.sum())

cpe imputed: 93 | values used: {39.0: 92, 0.02: 1}
total_spend imputed (excluded from spend training and evaluation): 3
spend_near_zero (excluded from MAPE): 8
budget_anomaly rows: 8


## Step 6. npi_specialty and region (DQ-25, DQ-26, DQ-27)

In [7]:
df["npi_specialty"] = df["npi_specialty"].replace({"3320 - Anesthesiologists": "Anesthesiology"})
df["region"] = df["region"].replace({"West South Central": "Southwest"})
df["region_country_mismatch"] = (~df["country"].isin(["USA", "UNKNOWN"])).astype(int)
print(df["npi_specialty"].value_counts().to_dict())
print(df["region"].value_counts().to_dict())
print("region_country_mismatch rows:", df.region_country_mismatch.sum())

{'Nephrology': 48, 'Cardiology': 40, 'Primary Care': 36, 'Internal Medicine': 35, 'Rheumatology': 33, 'Endocrinology': 31, 'Oncology': 31, 'Dermatology': 30, 'Hematology': 30, 'Gastroenterology': 28, 'Pulmonology': 27, 'Neurology': 24, 'Urology': 23, 'Anesthesiology': 1}
{'Mountain': 64, 'South Atlantic': 51, 'Mid-Atlantic': 50, 'Northeast': 50, 'Southwest': 47, 'Pacific': 41, 'West': 40, 'Southeast': 39, 'Midwest': 35}
region_country_mismatch rows: 160


## Step 7. Engineered numeric features (DQ-16, DQ-21, DQ-34)
Log versions of the heavily skewed columns, including `log_total_spend`, which is the version of the primary target we will model. Plus `n_periods` (the divisor linking total and average NRx). Original columns are kept alongside.

In [8]:
LOG_COLS = ["unique_reach", "unique_engaged_hcps", "total_engagements", "total_budget_spent",
            "total_spend", "actual_spend_per_engagement"]
for c in LOG_COLS:
    df[f"log_{c}"] = np.log1p(df[c])
df["n_periods"] = (df["pre_NRx"] / df["pre_avg_NRx"]).round(2)
df[[f"log_{c}" for c in LOG_COLS]].skew().round(2).to_frame("skew after log1p")

,skew after log1p
log_unique_reach,0.30
log_unique_engaged_hcps,0.32
log_total_engagements,0.28
log_total_budget_spent,-1.09
log_total_spend,-2.48
log_actual_spend_per_engagement,-3.24


## Step 8. Final column order and save `cleaned_df.csv`
Columns are grouped so the targets and the columns that must never be model inputs are easy to spot:

| Group | Columns | Why |
|---|---|---|
| Targets | `total_spend`, `log_total_spend`, `cpe` | What we predict |
| Spend-derived (DQ-35) | `actual_spend_per_engagement`, `log_actual_spend_per_engagement`, `spend_per_NRx_lift` | Calculated from total_spend, so they rebuild the target |
| Post-campaign (DQ-22, 23, 28, 29) | `post_NRx`, `post_avg_NRx`, `NRx_lift`, `NRx_lift_pct` | Measured after the campaign, not known when planning H2 |

In [9]:
ID_COLS = ["brand_raw", "company", "product", "channel_tag"]
CAT_COLS = ["sub_product_name", "country", "country_name", "pillar", "media_type",
            "profession", "specialty", "npi_specialty", "region"]
DATE_COLS = ["campaign_start_date", "start_year", "start_quarter", "start_month", "campaign_age_days"]
ENGAGE_COLS = ["unique_reach", "unique_engaged_hcps", "total_engagements", "engagement_rate", "engagement_frequency"]
PRE_COLS = ["pre_NRx", "pre_avg_NRx", "pre_NRx_avg", "n_periods"]
BUDGET_COLS = ["total_budget_spent"]
LOG_INPUTS = ["log_unique_reach", "log_unique_engaged_hcps", "log_total_engagements", "log_total_budget_spent"]
TARGET_COLS = ["total_spend", "log_total_spend", "cpe"]
SPEND_DERIVED_COLS = ["actual_spend_per_engagement", "log_actual_spend_per_engagement", "spend_per_NRx_lift"]
POST_CAMPAIGN_COLS = ["post_NRx", "post_avg_NRx", "NRx_lift", "NRx_lift_pct"]
FLAG_COLS = ["targeting_info_missing", "has_specialty", "start_date_imputed", "cpe_imputed",
             "total_spend_imputed", "spend_near_zero", "budget_anomaly", "region_country_mismatch"]

ORDER = (ID_COLS + CAT_COLS + DATE_COLS + ENGAGE_COLS + PRE_COLS + BUDGET_COLS + LOG_INPUTS
         + TARGET_COLS + SPEND_DERIVED_COLS + POST_CAMPAIGN_COLS + FLAG_COLS)
assert set(ORDER) == set(df.columns), set(df.columns) ^ set(ORDER)
cleaned_df = df[ORDER].copy()
cleaned_df["campaign_start_date"] = cleaned_df["campaign_start_date"].dt.strftime("%Y-%m-%d")

cleaned_df.to_csv(OUT_CLEAN, index=False)
print("Saved", OUT_CLEAN, cleaned_df.shape)
cleaned_df.head()

Saved ../data/processed/cleaned_df.csv (417, 50)


,brand_raw,company,product,channel_tag,sub_product_name,country,country_name,pillar,media_type,profession,specialty,npi_specialty,region,campaign_start_date,start_year,start_quarter,start_month,campaign_age_days,unique_reach,unique_engaged_hcps,total_engagements,engagement_rate,engagement_frequency,pre_NRx,pre_avg_NRx,pre_NRx_avg,n_periods,total_budget_spent,log_unique_reach,log_unique_engaged_hcps,log_total_engagements,log_total_budget_spent,total_spend,log_total_spend,cpe,actual_spend_per_engagement,log_actual_spend_per_engagement,spend_per_NRx_lift,post_NRx,post_avg_NRx,NRx_lift,NRx_lift_pct,targeting_info_missing,has_specialty,start_date_imputed,cpe_imputed,total_spend_imputed,spend_near_zero,budget_anomaly,region_country_mismatch
0,ABBVIE,ABBVIE,unspecified,none,BRAND ALERT,USA,United States,search,sponsored_videos,code_8,unknown,Neurology,Southeast,2024-07-31,2024,3,7,625,9270,1070,4449,0.1154,4.16,946.88,63.0600,47.4714,15.02,173511.0,9.134647,6.976348,8.400659,12.064002,134355308.0,18.715998,39.0,30198.99,10.315597,915440.73,1093.64,72.8343,146.77,15.5000,0,0,0,0,0,0,0,0
1,ABBVIE,ABBVIE,unspecified,none,BRAND ALERT,USA,United States,search,editorial_videos,code_8,unknown,Cardiology,Southwest,2024-08-19,2024,3,8,606,1959,380,1178,0.1940,3.10,915.49,48.2253,62.6833,18.98,45942.0,7.580700,5.942799,7.072422,10.735157,10667850.0,16.182745,39.0,9055.90,9.111282,95202.91,1027.54,54.1279,112.05,12.2398,0,0,0,0,0,0,0,0
2,ABBVIE,ABBVIE,unspecified,none,BRAND ALERT,USA,United States,search,sponsored_videos,psychologist,urology,Cardiology,Mid-Atlantic,2023-11-16,2023,4,11,883,2860,403,1703,0.1409,4.23,721.23,37.9804,66.2213,18.99,66417.0,7.958926,6.001415,7.440734,11.103723,59103000.0,17.894792,39.0,34705.23,10.454674,755161.42,799.50,42.1019,78.27,10.8516,0,1,0,0,0,0,0,0
3,ABBVIE,ABBVIE,unspecified,none,BRAND ALERT,ZAF,South Africa,search,sponsored_videos,code_8,unknown,Neurology,Northeast,2024-11-22,2024,4,11,511,7390,910,1107,0.1231,1.22,1022.39,44.2658,55.8293,23.10,43173.0,8.908018,6.814543,7.010312,10.672994,124487739.0,18.639718,39.0,112455.05,11.630318,1021479.17,1144.26,49.5424,121.87,11.9201,0,0,0,0,0,0,0,1
4,ABBVIE,ABBVIE,unspecified,none,EMAIL ALERT,UNKNOWN,Unknown,unknown,unknown,unknown,unknown,Endocrinology,Southwest,2023-11-16,2023,4,11,883,77570,26264,116027,0.3386,4.42,630.86,34.7051,39.3346,18.18,348081.0,11.258949,10.175993,11.661587,12.760193,892409925.0,20.609436,3.0,7691.40,8.947988,4028833.69,852.37,46.8907,221.51,35.1116,1,0,0,0,0,0,0,0


## Step 9. Encoding for modeling (`cleaned_df_encoded.csv`)
- **One-hot encoding** for all nominal categories. None of these columns have a natural order, so label or ordinal encoding would invent a ranking that does not exist.
- **Rare-category grouping:** any category with fewer than 5 rows is merged into `OTHER` before encoding, so we do not create columns that are nearly always zero. Company uses the same rule. (When we split train and test in October, this grouping can be recomputed on the training rows only.)
- `product` is reduced to a `has_named_product` flag because most products appear only once or twice.
- Raw text and duplicate label columns (`brand_raw`, `country_name`, `product`, `campaign_start_date`) are removed.
- All numeric columns and flags are kept as is. Scaling is left to the modeling pipeline so it can be fit on training data only.

In [10]:
MIN_COUNT = 5
enc = cleaned_df.copy()
enc["has_named_product"] = (enc["product"] != "unspecified").astype(int)

ONEHOT_COLS = ["company", "channel_tag", "sub_product_name", "country", "pillar", "media_type",
               "profession", "specialty", "npi_specialty", "region"]
for c in ONEHOT_COLS:
    counts = enc[c].value_counts()
    rare = counts[counts < MIN_COUNT].index
    enc[c] = enc[c].where(~enc[c].isin(rare), "OTHER")

enc = enc.drop(columns=["brand_raw", "country_name", "product", "campaign_start_date"])
enc = pd.get_dummies(enc, columns=ONEHOT_COLS, prefix=ONEHOT_COLS, dtype=int)
enc.columns = (enc.columns.str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True).str.strip("_"))
assert enc.columns.is_unique

enc.to_csv(OUT_ENCODED, index=False)
print("Saved", OUT_ENCODED, enc.shape)
print("One-hot columns created per source column:")
print({c: sum(col.startswith(c + "_") and col not in FLAG_COLS for col in enc.columns) for c in ONEHOT_COLS})

Saved ../data/processed/cleaned_df_encoded.csv (417, 124)
One-hot columns created per source column:
{'company': 22, 'channel_tag': 5, 'sub_product_name': 10, 'country': 7, 'pillar': 5, 'media_type': 3, 'profession': 8, 'specialty': 4, 'npi_specialty': 14, 'region': 9}


## Step 10. Feature sets for each target
This is the documented feature set the milestone asks for. Every modeling notebook should load its columns from `reports/feature_sets.json` instead of picking them by hand, so no one accidentally includes a leaking column.

**Never used as features (either target):** `total_spend` and `log_total_spend`, the spend-derived columns, the post-campaign columns, and the two flags built from the spend target (`total_spend_imputed`, `spend_near_zero`).

| Target | Extra columns left out | Rows used for training | Rows used for MAPE |
|---|---|---|---|
| `log_total_spend` (primary) | none beyond the list above. `cpe` and `total_budget_spent` are planned before a campaign runs, so they are fair inputs. | `total_spend_imputed == 0` (414 rows) | also `spend_near_zero == 0` |
| `cpe` (secondary) | `cpe` itself, plus `total_budget_spent`, `log_total_budget_spent`, `cpe_imputed`, `budget_anomaly` (DQ-19: budget / engagements rebuilds CPE) | all 417 rows | all rows |

**Note:** the team README also lists engagement volume as a possible target. If we model it, the engagement columns (`total_engagements`, `unique_engaged_hcps`, `engagement_rate`, `engagement_frequency` and their logs) would need to come out of that feature set the same way.

In [11]:
import json

NEVER_FEATURES = set(["total_spend", "log_total_spend"] + SPEND_DERIVED_COLS + POST_CAMPAIGN_COLS + ["total_spend_imputed", "spend_near_zero"])
CPE_EXTRA_EXCLUDE = {"cpe", "total_budget_spent", "log_total_budget_spent", "cpe_imputed", "budget_anomaly"}

spend_features = [c for c in enc.columns if c not in NEVER_FEATURES]
cpe_features = [c for c in spend_features if c not in CPE_EXTRA_EXCLUDE]

feature_sets = {
    "notes": "Column names refer to data/processed/cleaned_df_encoded.csv. Generated by notebooks/02_data_cleaning.ipynb.",
    "total_spend": {
        "role": "primary target",
        "target_column": "log_total_spend",
        "report_on_scale": "total_spend (convert predictions back with np.expm1)",
        "metrics": ["MAPE (goal under 30%)", "R2 (goal above 0.55)"],
        "train_rows": "total_spend_imputed == 0",
        "mape_rows": "total_spend_imputed == 0 and spend_near_zero == 0",
        "features": spend_features,
    },
    "cpe": {
        "role": "secondary target (confirm meta_mapped_cpe with Challenge Advisor)",
        "target_column": "cpe",
        "train_rows": "all rows",
        "features": cpe_features,
    },
    "never_features": sorted(NEVER_FEATURES),
}
with open(OUT_FEATURES, "w") as f:
    json.dump(feature_sets, f, indent=2)

print(f"Saved {OUT_FEATURES}")
print(f"total_spend: {len(spend_features)} features | training rows: {(cleaned_df.total_spend_imputed == 0).sum()} | "
      f"MAPE rows: {((cleaned_df.total_spend_imputed == 0) & (cleaned_df.spend_near_zero == 0)).sum()}")
print(f"cpe: {len(cpe_features)} features | training rows: {len(cleaned_df)}")

Saved ../reports/feature_sets.json
total_spend: 113 features | training rows: 414 | MAPE rows: 406
cpe: 108 features | training rows: 417


## Step 11. Validation checks

In [12]:
checks = {
    "row count unchanged (417)": len(cleaned_df) == len(raw) == len(enc),
    "no nulls in cleaned_df": cleaned_df.isna().sum().sum() == 0,
    "no nulls in encoded": enc.isna().sum().sum() == 0,
    "encoded is fully numeric": all(pd.api.types.is_numeric_dtype(t) for t in enc.dtypes),
    "no 'undefined' left in specialty": not cleaned_df["specialty"].eq("undefined").any(),
    "campaign_end_date removed": "campaign_end_date" not in cleaned_df.columns,
    "no 2099 dates remain": not cleaned_df["campaign_start_date"].str.startswith("2099").any(),
    "cpe x engagements = budget (non-anomaly rows)": (
        (cleaned_df.cpe * cleaned_df.total_engagements - cleaned_df.total_budget_spent).abs()[cleaned_df.budget_anomaly == 0] <= 1).all(),
    "every brand mapped": cleaned_df["company"].notna().all(),
    "one company per raw brand": cleaned_df.groupby("brand_raw")["company"].nunique().max() == 1,
    "no spend target or leaking column in spend features": not (set(spend_features) & NEVER_FEATURES),
    "no cpe, budget or leaking column in cpe features": not (set(cpe_features) & (NEVER_FEATURES | CPE_EXTRA_EXCLUDE)),
    "every feature exists in encoded file": set(spend_features) <= set(enc.columns),
}
pd.Series(checks, name="passes").to_frame()

,passes
row count unchanged (417),True
no nulls in cleaned_df,True
no nulls in encoded,True
encoded is fully numeric,True
no 'undefined' left in specialty,True
campaign_end_date removed,True
no 2099 dates remain,True
cpe x engagements = budget (non-anomaly rows),True
every brand mapped,True
one company per raw brand,True


In [13]:
assert all(checks.values()), "A validation check failed"
print("All checks passed.")

All checks passed.


## Summary of cleaning decisions
| Issue | Decision |
|---|---|
| DQ-01 brand | Mapped 73 raw values into `company`, `product`, `channel_tag`. Raw value kept as `brand_raw`. |
| DQ-03, 04, 06, 07, 08 nulls in country, pillar, media_type | Filled with `unknown` / `UNKNOWN`. Added `targeting_info_missing` flag for the 56-row block. No rows dropped. |
| DQ-09 to DQ-13 profession, specialty | Numeric codes stored as `code_N` labels (nominal, not ordinal). `undefined` treated as missing. Added `has_specialty`. |
| DQ-14 start date | Parsed to a date. 3 nulls filled with company median and flagged. Year, quarter, month and age features added. |
| DQ-15 end date | Dropped (100% placeholder or null). |
| DQ-17 total_engagements | Not null in this file, so it is kept. The drop in our original plan is not needed. |
| DQ-18 budget | 8 anomalous rows flagged with `budget_anomaly`, not removed. |
| DQ-19 CPE leakage | `total_budget_spent` and its log left out of the CPE feature set. |
| DQ-25, 26 npi_specialty, region | One label fixed in each. `region_country_mismatch` flag added. |
| DQ-16, 34 skew | log1p features added, including `log_total_spend` as the modeling target. Outliers kept. |
| DQ-22, 23, 28, 29 post-campaign | Kept for EDA, left out of both feature sets. |
| DQ-30 cpe nulls | 93 filled from budget / engagements (exact). Flagged. 92 come out at 39, matching the BRAND ALERT rate. |
| DQ-31 CPE target | Known cpe is one rate per product. Kept as secondary target pending confirmation of `meta_mapped_cpe`. |
| DQ-32 total_spend nulls | Filled so the file has no blanks, flagged, and excluded from spend training and evaluation (true value unknown). |
| DQ-33 near-zero spend | 8 rows flagged with `spend_near_zero` and excluded from MAPE. |
| DQ-35 spend leakage | Spend-derived columns left out of both feature sets. |
| Encoding | One-hot for all nominal columns, categories under 5 rows grouped into `OTHER`. |